# Frequency Branch — Complete Final Notebook

## How this works
1. **Extraction cell** — reads parquet files once, writes wanted images to disk as JPEG, releases memory after every file. ~20 min, ~1.5GB disk.
2. **Split cell** — builds train/val/test from all JPEG paths (StyleGAN + Flickr + DeepDetect + OpenFake extracted). No parquet at training time.
3. **Training** — pure JPEG loading, fast, no memory issues.

## Why together is better than phases
Training on all fake types simultaneously means the model learns a unified real-vs-fake boundary. Phase-wise training causes the model to overfit GAN patterns first, then struggle to generalise to diffusion models in phase 2.

## Memory guarantees
- Extraction: peak ~400MB RAM per file, released immediately after
- Training: no parquet, no accumulation, workers load JPEG only

In [1]:
import os, io, gc, glob, random, csv
from pathlib import Path
from dataclasses import dataclass
from collections import Counter, defaultdict

import numpy as np
from PIL import Image, ImageFilter
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print('Imports OK')

Imports OK


In [2]:
# ══════════════════════════════════════════════
#  CONFIG
# ══════════════════════════════════════════════
SEED         = 42
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'

# training
FORCE_POS_WEIGHT = 1.5 
START_EPOCH  = 7
EPOCHS       = 15
BATCH_SIZE   = 32
LR           = 1e-4
LR_MIN       = 1e-6
WEIGHT_DECAY = 1e-4
GRAD_CLIP    = 1.0

# image
TARGET_SIZE  = 224
RESIZE_SIZE  = 256
FFT_CLIP_MAX = 14.0
IMG_EXTS     = {'.jpg', '.jpeg', '.png', '.webp'}

# path-dataset caps
CAPS = {
    'stylegan_real':   50000,
    'stylegan_fake':   50000,
    'flickr_real':     30000,
    'deepdetect_fake': 30000,
}

# OpenFake extraction
# 1500 per generator x 33 gens = 49500 fake + 49500 real = ~99K samples
# Disk: ~99K x 15KB = ~1.5GB  (fits in Kaggle 20GB working dir)
# Each generator has 1500 samples — strong enough representation.
# Going below 1000 risks generators with <1000 total images being partial.
OF_PER_GEN   = 1500
EXTRACT_ROOT = Path('/kaggle/working/of_imgs')
MANIFEST     = '/kaggle/working/of_manifest.csv'

# fast val subset per epoch
FAST_VAL_N   = 4000

# checkpoints
CKPT_LOAD    =    '/kaggle/input/datasets/frequency-model-checkpoint/best_model.pth'
CKPT_DIR     = '/kaggle/working'

# seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

print('Device:', DEVICE)
print('Config OK')

Device: cuda
Config OK


In [3]:
# ══════════════════════════════════════════════
#  PATHS
# ══════════════════════════════════════════════
STYLEGAN_ROOT   = Path('/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train')
FLICKR_ROOT     = Path('/kaggle/input/datasets/adityajn105/flickr30k/Images/flickr30k_images')
DEEPDETECT_ROOT = Path('/kaggle/input/datasets/ayushmandatta1/deepdetect-2025/ddata/train')
OPENFAKE_DIRS   = [
    Path('/kaggle/input/datasets/kashirhanif/openfake-part-1'),
    Path('/kaggle/input/datasets/kashirhanif/openfake-part-2'),
    Path('/kaggle/input/datasets/kashirhanif/openfake-part-3'),
    Path('/kaggle/input/datasets/kashirhanif/openfake-part-4'),
]

for name, p in [('StyleGAN', STYLEGAN_ROOT), ('Flickr', FLICKR_ROOT), ('DeepDetect', DEEPDETECT_ROOT)]:
    print(f'  {name}: {"OK" if p.exists() else "MISSING"}  ({p})')
for i, p in enumerate(OPENFAKE_DIRS):
    print(f'  OpenFake-{i+1}: {"OK" if p.exists() else "MISSING"}')

  StyleGAN: OK  (/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train)
  Flickr: OK  (/kaggle/input/datasets/adityajn105/flickr30k/Images/flickr30k_images)
  DeepDetect: OK  (/kaggle/input/datasets/ayushmandatta1/deepdetect-2025/ddata/train)
  OpenFake-1: OK
  OpenFake-2: OK
  OpenFake-3: OK
  OpenFake-4: OK


In [4]:
# ══════════════════════════════════════════════
#  OPENFAKE EXTRACTION
#  Run once. If manifest already exists, skips automatically.
#
#  Memory design:
#  - pq.read_table() reads one file fully then releases on del
#  - We read label+model (cheap), find wanted row indices,
#    then read image column (expensive) only for that file
#  - del + gc.collect() after every file = peak ~400MB, then free
#  - Images written immediately as JPEG — never accumulate in RAM
# ══════════════════════════════════════════════
import pyarrow.parquet as pq

OF_GENERATORS = [
    'sd-3.5','sd-2.1','flux.1-dev','sdxl-epic-realism','midjourney-6',
    'gpt-image-1','sdxl','sd-1.5-dreamshaper','flux.1-schnell','dalle-3',
    'sdxl-touchofrealism','flux-1.1-pro','sdxl-realvis-v5','ideogram-3.0',
    'hidream-i1-full','sdxl-juggernaut','sd-1.5-epicdream','sd-1.5',
    'flux-mvc5000','mystic','imagen-4.0','grok-2-image-1212','chroma',
    'flux-amateursnapshotphotos','imagen-3.0-002','midjourney-7',
    'flux-realism','recraft-v3','lumina-17-2-25','recraft-v2',
    'aurora-20-1-25','ideogram-2.0','frames-23-1-25',
]
OF_REAL_CAP = OF_PER_GEN * len(OF_GENERATORS)  # balanced


def extract_openfake():
    if os.path.exists(MANIFEST):
        n = sum(1 for _ in open(MANIFEST)) - 1
        print(f'Manifest exists: {n:,} images. Delete {MANIFEST} to re-run.')
        return

    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

    need    = {g: OF_PER_GEN for g in OF_GENERATORS}
    need['real'] = OF_REAL_CAP
    written = defaultdict(int)
    rows    = []
    gidx    = 0

    parquet_files = sorted(
        f for d in OPENFAKE_DIRS
        for f in glob.glob(str(d / '*.parquet'))
    )
    print(f'{len(parquet_files)} parquet files found.')
    total_need = sum(need.values())
    print(f'Extracting ~{total_need:,} images total  (~{total_need*15//1024} MB)')

    for fp in tqdm(parquet_files, desc='Extracting'):
        if all(written.get(g, 0) >= need[g] for g in need):
            print('All quotas filled.')
            break

        # ── Step 1: read metadata only (very cheap) ──
        try:
            meta = pq.read_table(fp, columns=['label', 'model'])
        except Exception as e:
            print(f'  Skip {Path(fp).name}: {e}')
            continue

        labels = meta.column('label').to_pylist()
        models = meta.column('model').to_pylist()
        del meta
        gc.collect()

        wanted = []
        for i, (lbl, mdl) in enumerate(zip(labels, models)):
            g = 'real' if lbl == 'real' else str(mdl)
            if g not in need:
                continue
            if written.get(g, 0) >= need[g]:
                continue
            wanted.append((i, g, 0 if g == 'real' else 1))

        del labels, models
        gc.collect()

        if not wanted:
            continue

        # ── Step 2: read images for wanted rows ──
        try:
            img_tbl = pq.read_table(fp, columns=['image'])
        except Exception as e:
            print(f'  Skip images {Path(fp).name}: {e}')
            continue

        img_col = img_tbl.column('image')
        
        for (row_i, g, y) in wanted:
            if written.get(g, 0) >= need[g]:
                continue
            try:
                raw = img_col[row_i].as_py()
                if isinstance(raw, dict) and 'bytes' in raw:
                    raw = raw['bytes']
                
                img = Image.open(io.BytesIO(raw)).convert('RGB')
                safe   = g.replace('/','_').replace('.','_').replace('-','_')
                folder = EXTRACT_ROOT / safe
                folder.mkdir(exist_ok=True)
                out = folder / f'{gidx:08d}.jpg'
                img.save(str(out), format='JPEG', quality=88, optimize=True)
                img.close()        
                rows.append(f'{out},{y},{g}')
                written[g] += 1
                gidx += 1
                
                    
            except Exception:
                gidx += 1

        # ── Critical: release image data before next file ──
        del img_tbl, img_col
        gc.collect()

    # Write manifest
    with open(MANIFEST, 'w') as f:
        f.write('path,label,source\n')
        f.write('\n'.join(rows))

    print(f'\nDone: {gidx:,} images written to {EXTRACT_ROOT}')
    print(f'Manifest: {MANIFEST}')
    short = [(g, written.get(g,0), need[g]) for g in need if written.get(g,0) < need[g]]
    if short:
        print('Partial (ran out of source images):')
        for g, got, req in short:
            print(f'  {g}: {got}/{req}')
    else:
        print('All generators fully extracted.')


extract_openfake()

89 parquet files found.
Extracting ~99,000 images total  (~1450 MB)


Extracting:  10%|█         | 9/89 [28:22<4:12:09, 189.12s/it]


KeyboardInterrupt: 

In [5]:
# ══════════════════════════════════════════════
#  SAMPLE REFERENCE
# ══════════════════════════════════════════════
@dataclass(frozen=True)
class SampleRef:
    path:   str
    label:  int
    source: str


def list_images(p: Path):
    if not p.exists():
        return []
    return [x for x in p.rglob('*') if x.is_file() and x.suffix.lower() in IMG_EXTS]


def cap_shuffle(paths, cap):
    paths = [p for p in paths if p.suffix.lower() in IMG_EXTS]
    random.shuffle(paths)
    return paths[:cap] if cap else paths


def split_paths(paths, label, source, tr=0.75, va=0.10):
    random.shuffle(paths)
    n  = len(paths)
    nt = int(tr * n)
    nv = int(va * n)
    mk = lambda ps: [SampleRef(str(p), label, source) for p in ps]
    return mk(paths[:nt]), mk(paths[nt:nt+nv]), mk(paths[nt+nv:])


def split_refs_per_source(refs, tr=0.75, va=0.10):
    """Split per source so every generator appears in all three splits."""
    by_src = defaultdict(list)
    for r in refs:
        by_src[r.source].append(r)
    train_out, val_out, test_out = [], [], []
    for src_refs in by_src.values():
        random.shuffle(src_refs)
        n  = len(src_refs)
        nt = int(tr * n)
        nv = int(va * n)
        train_out += src_refs[:nt]
        val_out   += src_refs[nt:nt+nv]
        test_out  += src_refs[nt+nv:]
    return train_out, val_out, test_out


print('SampleRef helpers defined.')

SampleRef helpers defined.


In [6]:
# ══════════════════════════════════════════════
#  BUILD SPLITS — all JPEG paths
# ══════════════════════════════════════════════
print('Listing images...')
sty_real = cap_shuffle(list_images(STYLEGAN_ROOT / 'real'), CAPS['stylegan_real'])
sty_fake = cap_shuffle(list_images(STYLEGAN_ROOT / 'fake'), CAPS['stylegan_fake'])
flk_real = cap_shuffle(list_images(FLICKR_ROOT),            CAPS['flickr_real'])
dd_fake  = cap_shuffle(list_images(DEEPDETECT_ROOT / 'fake'), CAPS['deepdetect_fake'])

print(f'  StyleGAN real: {len(sty_real):,}')
print(f'  StyleGAN fake: {len(sty_fake):,}')
print(f'  Flickr real:   {len(flk_real):,}')
print(f'  DeepDetect fk: {len(dd_fake):,}')

train_refs, val_refs, test_refs = [], [], []
for paths, lbl, src in [
    (sty_real, 0, 'stylegan_real'),
    (sty_fake, 1, 'stylegan_fake'),
    (flk_real, 0, 'flickr_real'),
    (dd_fake,  1, 'deepdetect_fake'),
]:
    tr, va, te = split_paths(paths, lbl, src)
    train_refs += tr
    val_refs   += va
    test_refs  += te

# Load OpenFake manifest if it exists
if os.path.exists(MANIFEST):
    of_refs = []
    with open(MANIFEST, 'r') as f:
        for row in csv.DictReader(f):
            g   = row['source']
            src = 'openfake_real' if g == 'real' else f'openfake_{g}'
            of_refs.append(SampleRef(path=row['path'], label=int(row['label']), source=src))
    print(f'\nOpenFake manifest: {len(of_refs):,} images')

    of_tr, of_va, of_te = split_refs_per_source(of_refs)
    print(f'OpenFake splits — train: {len(of_tr):,}  val: {len(of_va):,}  test: {len(of_te):,}')
    train_refs += of_tr
    val_refs   += of_va
    test_refs  += of_te
else:
    print('\nNo manifest found — using path-only datasets only.')
    print('Run the extraction cell to add OpenFake.')

# Final shuffle
random.shuffle(train_refs)
random.shuffle(val_refs)
random.shuffle(test_refs)

print(f'\nFinal splits:')
print(f'  Train: {len(train_refs):,}')
print(f'  Val:   {len(val_refs):,}')
print(f'  Test:  {len(test_refs):,}')

tc = Counter(r.label for r in train_refs)
print(f'  Real: {tc[0]:,} ({100*tc[0]/len(train_refs):.1f}%)  Fake: {tc[1]:,} ({100*tc[1]/len(train_refs):.1f}%)')

Listing images...
  StyleGAN real: 50,000
  StyleGAN fake: 50,000
  Flickr real:   30,000
  DeepDetect fk: 30,000

OpenFake manifest: 92,532 images
OpenFake splits — train: 69,396  val: 9,252  test: 13,884

Final splits:
  Train: 189,396
  Val:   25,252
  Test:  37,884
  Real: 97,125 (51.3%)  Fake: 92,271 (48.7%)


In [7]:
# ══════════════════════════════════════════════
#  FFT TRANSFORM
# ══════════════════════════════════════════════
def img_to_fft_tensor(img: Image.Image) -> torch.Tensor:
    arr = np.array(img, dtype=np.float32) / 255.0
    x   = torch.from_numpy(arr).permute(2, 0, 1)
    out = []
    for c in range(3):
        f   = torch.fft.fftshift(torch.fft.fft2(x[c]))
        mag = torch.log1p(torch.abs(f))
        mag = torch.clamp(mag, 0.0, FFT_CLIP_MAX) / FFT_CLIP_MAX
        out.append(mag)
    return torch.stack(out)


def center_crop(img):
    img  = img.convert('RGB')
    img  = img.resize((RESIZE_SIZE, RESIZE_SIZE), Image.BICUBIC)
    left = (RESIZE_SIZE - TARGET_SIZE) // 2
    return img.crop((left, left, left + TARGET_SIZE, left + TARGET_SIZE))


def augment(img):
    if random.random() < 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
    if random.random() < 0.15:
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.1, 0.5)))
    if random.random() < 0.4:
        buf = io.BytesIO()
        img.save(buf, format='JPEG', quality=random.randint(60, 95))
        buf.seek(0)
        img = Image.open(buf).convert('RGB')
    return img


print('FFT transform defined.')

FFT transform defined.


In [8]:
# ══════════════════════════════════════════════
#  DATASET
# ══════════════════════════════════════════════
class FFTDataset(Dataset):
    def __init__(self, refs, train=True):
        self.refs  = refs
        self.train = train

    def __len__(self):
        return len(self.refs)

    def __getitem__(self, idx):
        r = self.refs[idx]
        try:
            img = Image.open(r.path).convert('RGB')
        except Exception:
            return (torch.zeros(3, TARGET_SIZE, TARGET_SIZE),
                    torch.tensor(0.0), r.source)
        img = center_crop(img)
        if self.train:
            img = augment(img)
        return img_to_fft_tensor(img), torch.tensor(float(r.label)), r.source


print('FFTDataset defined.')

FFTDataset defined.


In [9]:
# ══════════════════════════════════════════════
#  DATALOADERS
# ══════════════════════════════════════════════
fast_val_refs = random.sample(val_refs, min(FAST_VAL_N, len(val_refs)))

train_ds    = FFTDataset(train_refs,    train=True)
fast_val_ds = FFTDataset(fast_val_refs, train=False)
val_ds      = FFTDataset(val_refs,      train=False)
test_ds     = FFTDataset(test_refs,     train=False)

KW = dict(num_workers=4, pin_memory=True, persistent_workers=True)

train_loader    = DataLoader(train_ds,    batch_size=BATCH_SIZE, shuffle=True,  **KW)
fast_val_loader = DataLoader(fast_val_ds, batch_size=BATCH_SIZE, shuffle=False, **KW)
val_loader      = DataLoader(val_ds,      batch_size=BATCH_SIZE, shuffle=False, **KW)
test_loader     = DataLoader(test_ds,     batch_size=BATCH_SIZE, shuffle=False, **KW)

print(f'train:    {len(train_ds):,}  ({len(train_loader):,} batches)')
print(f'fast_val: {len(fast_val_ds):,}')
print(f'val:      {len(val_ds):,}')
print(f'test:     {len(test_ds):,}')

x, y, src = next(iter(train_loader))
print(f'\nBatch: {x.shape}  range [{x.min():.3f}, {x.max():.3f}]  labels {y.unique().tolist()}')
del x, y, src

train:    189,396  (5,919 batches)
fast_val: 4,000
val:      25,252
test:     37,884

Batch: torch.Size([32, 3, 224, 224])  range [0.000, 0.753]  labels [0.0, 1.0]


In [10]:
# ══════════════════════════════════════════════
#  MODEL
# ══════════════════════════════════════════════
def build_model():
    m = efficientnet_b3(weights=EfficientNet_B3_Weights.IMAGENET1K_V1)
    m.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(m.classifier[1].in_features, 1)
    )
    return m

model = build_model().to(DEVICE)
print(f'EfficientNet-B3: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params')

Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 184MB/s]


EfficientNet-B3: 10.7M params


In [11]:
# ══════════════════════════════════════════════
#  LOSS / OPTIMIZER / SCHEDULER
#  Fix: forced pos_weight + macro F1 for best-model selection
# ══════════════════════════════════════════════
tc    = Counter(r.label for r in train_refs)
ratio = tc[0] / max(tc[1], 1)
print(f'real: {tc[0]:,}  fake: {tc[1]:,}  ratio: {ratio:.3f}')

# Always apply pos_weight — even at ratio=1.0 this prevents
# the fake-prediction collapse seen in the previous run.
# A value of 1.5 means missing a real image costs 1.5x more than
# missing a fake image, keeping the model honest on both classes.
pos_w = torch.tensor([max(ratio, FORCE_POS_WEIGHT)],
                     dtype=torch.float32, device=DEVICE)
print(f'pos_weight = {pos_w.item():.3f}')

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=max(1, EPOCHS - START_EPOCH), eta_min=LR_MIN
)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))
print(f'AdamW lr={LR}  CosineAnnealing T_max={EPOCHS-START_EPOCH}')

real: 97,125  fake: 92,271  ratio: 1.053
pos_weight = 1.500
AdamW lr=0.0001  CosineAnnealing T_max=8


In [13]:
# ══════════════════════════════════════════════
#  RESUME (optional)
# ══════════════════════════════════════════════
best_val_f1 = 0.0

if CKPT_LOAD and os.path.exists(CKPT_LOAD):
    ckpt = torch.load(CKPT_LOAD, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    if 'scheduler_state_dict' in ckpt:
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    START_EPOCH = ckpt.get('epoch', 0)
    best_val_f1 = ckpt.get('best_val_f1', 0.0)
    print(f'Resumed epoch {START_EPOCH}, best_f1={best_val_f1:.4f}')
else:
    print('Starting from scratch.')

Starting from scratch.


In [19]:
THRESHOLD = 0.35

@torch.no_grad()
def evaluate(model, loader, desc='Eval', threshold=THRESHOLD):
    model.eval()
    pa, la, sa = [], [], []
    all_probs = []

    for x, y, src in tqdm(loader, desc=desc, leave=False):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
            logits = model(x).squeeze(1)
        probs = torch.sigmoid(logits)
        preds = (probs >= threshold).long()
        pa.extend(preds.cpu().tolist())
        la.extend(y.long().cpu().tolist())
        sa.extend(src if isinstance(src, (list, tuple)) else [src])
        all_probs.extend(probs.cpu().tolist())

    macro_f1 = f1_score(la, pa, average='macro', zero_division=0)

    ov = {
        'acc':      accuracy_score(la, pa),
        'macro_f1': macro_f1,
        'fake_f1':  f1_score(la, pa, zero_division=0),
        # Real F1: treat real(0) as the positive class
        'real_f1':  f1_score(la, pa, pos_label=0, zero_division=0),
        'fake_prec': precision_score(la, pa, zero_division=0),
        'fake_rec':  recall_score(la, pa, zero_division=0),
        'real_prec': precision_score(la, pa, pos_label=0, zero_division=0),
        'real_rec':  recall_score(la, pa, pos_label=0, zero_division=0),
        'mean_prob': sum(all_probs) / len(all_probs),
    }

    per = {}
    for s in set(sa):
        idx      = [i for i, ss in enumerate(sa) if ss == s]
        yt       = [la[i] for i in idx]
        yp       = [pa[i] for i in idx]
        is_real  = yt[0] == 0   # all samples in a source have the same label

        if is_real:
            # For real sources: measure how well we identify them AS real
            # pos_label=0 means "real" is the positive class
            per[s] = {
                'acc':  accuracy_score(yt, yp),
                'f1':   f1_score(yt, yp, pos_label=0, zero_division=0),
                'prec': precision_score(yt, yp, pos_label=0, zero_division=0),
                'rec':  recall_score(yt, yp, pos_label=0, zero_division=0),
                'n':    len(idx),
            }
        else:
            # For fake sources: measure how well we catch them AS fake
            per[s] = {
                'acc':  accuracy_score(yt, yp),
                'f1':   f1_score(yt, yp, pos_label=1, zero_division=0),
                'prec': precision_score(yt, yp, pos_label=1, zero_division=0),
                'rec':  recall_score(yt, yp, pos_label=1, zero_division=0),
                'n':    len(idx),
            }

    worst = min(m['f1'] for m in per.values()) if per else 0.0
    return ov, per, worst


def print_eval(tag, ov, per):
    print(f'\n{tag}')
    print(f'  acc={ov["acc"]:.4f}  macro_f1={ov["macro_f1"]:.4f}  mean_prob={ov["mean_prob"]:.4f}')
    print(f'  REAL  f1={ov["real_f1"]:.4f}  prec={ov["real_prec"]:.4f}  rec={ov["real_rec"]:.4f}')
    print(f'  FAKE  f1={ov["fake_f1"]:.4f}  prec={ov["fake_prec"]:.4f}  rec={ov["fake_rec"]:.4f}')
    print(f'\n  Per source (sorted by F1, real sources shown with real-as-positive):')
    real_srcs = {s: m for s, m in per.items() if m['f1'] > 0 or
                 any(r in s for r in ['stylegan_real','flickr_real','openfake_real'])}
    fake_srcs = {s: m for s, m in per.items() if s not in real_srcs}
    print(f'  -- Real sources --')
    for s, m in sorted(real_srcs.items(), key=lambda x: x[1]['f1']):
        print(f'  {s:40s}  acc={m["acc"]:.3f}  f1={m["f1"]:.3f}  '
              f'p={m["prec"]:.3f}  r={m["rec"]:.3f}  n={m["n"]}')
    print(f'  -- Fake sources --')
    for s, m in sorted(fake_srcs.items(), key=lambda x: x[1]['f1']):
        print(f'  {s:40s}  acc={m["acc"]:.3f}  f1={m["f1"]:.3f}  '
              f'p={m["prec"]:.3f}  r={m["rec"]:.3f}  n={m["n"]}')

print('evaluate() defined.')

evaluate() defined.


In [15]:
# ══════════════════════════════════════════════
#  DATALOADERS  —  stable settings for Kaggle
# ══════════════════════════════════════════════
fast_val_refs = random.sample(val_refs, min(FAST_VAL_N, len(val_refs)))

train_ds    = FFTDataset(train_refs,    train=True)
fast_val_ds = FFTDataset(fast_val_refs, train=False)
val_ds      = FFTDataset(val_refs,      train=False)
test_ds     = FFTDataset(test_refs,     train=False)

# Training loader: num_workers=4, no persistent_workers
# Validation loaders: num_workers=2, no persistent_workers
# persistent_workers=True caused worker death after epoch 3
train_loader    = DataLoader(train_ds,    batch_size=BATCH_SIZE, shuffle=True,
                             num_workers=4, pin_memory=True)
fast_val_loader = DataLoader(fast_val_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=2, pin_memory=True)
val_loader      = DataLoader(val_ds,      batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=2, pin_memory=True)
test_loader     = DataLoader(test_ds,     batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=2, pin_memory=True)

print(f'train:    {len(train_ds):,}  ({len(train_loader):,} batches)')
print(f'fast_val: {len(fast_val_ds):,}')
print(f'val:      {len(val_ds):,}')
print(f'test:     {len(test_ds):,}')

x, y, src = next(iter(train_loader))
print(f'Batch: {x.shape}  range [{x.min():.3f}, {x.max():.3f}]  labels {y.unique().tolist()}')
del x, y, src

train:    189,396  (5,919 batches)
fast_val: 4,000
val:      25,252
test:     37,884
Batch: torch.Size([32, 3, 224, 224])  range [0.000, 0.767]  labels [0.0, 1.0]


In [16]:
# ══════════════════════════════════════════════
#  TRAINING LOOP
# ══════════════════════════════════════════════
print(f'Training ep {START_EPOCH+1}->{EPOCHS}  |  {len(train_loader):,} batches/ep  |  bs={BATCH_SIZE}')
print(f'Estimated: ~{len(train_loader)*1.5/60:.0f} min/epoch')
print('=' * 60)

history = []

for epoch in range(START_EPOCH, EPOCHS):
    model.train()
    run_loss = run_correct = run_n = 0

    for x, y, _ in tqdm(train_loader, desc=f'Ep {epoch+1:02d}/{EPOCHS}'):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
            logits = model(x).squeeze(1)
            loss   = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        with torch.no_grad():
            run_correct += ((torch.sigmoid(logits) >= 0.5).long() == y.long()).sum().item()
        run_loss += loss.item() * x.size(0)
        run_n    += x.size(0)

    scheduler.step()
    tr_loss = run_loss / max(run_n, 1)
    tr_acc  = run_correct / max(run_n, 1)
    lr      = scheduler.get_last_lr()[0]

    fv, _, _ = evaluate(model, fast_val_loader, desc='FastVal')

    print(f'\nEp {epoch+1:02d}/{EPOCHS}'
          f'  loss={tr_loss:.4f}  tr_acc={tr_acc:.4f}'
          f'  macro_f1={fv["macro_f1"]:.4f}  real_acc={fv["real_acc"]:.4f}'
          f'  fake_acc={fv["fake_acc"]:.4f}  lr={lr:.2e}')

    ckpt = {
        'model_state_dict':     model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'epoch': epoch + 1, 'best_val_f1': best_val_f1,
        'train_loss': tr_loss, 'train_acc': tr_acc, 'val_f1': fv['f1'],
    }
    torch.save(ckpt, f'{CKPT_DIR}/epoch_{epoch+1:02d}.pth')
    print(f'  Saved epoch_{epoch+1:02d}.pth')

    # Use macro_f1 instead of fake-only f1 for best model selection
    # macro_f1 = average of real_F1 and fake_F1 — model must be good at BOTH
    if fv['macro_f1'] > best_val_f1:
        best_val_f1 = fv['macro_f1']
        torch.save(ckpt, f'{CKPT_DIR}/best_model.pth')
        print(f'  >> best_model.pth  macro_f1={best_val_f1:.4f}')

    history.append(dict(ep=epoch+1, loss=tr_loss, tr_acc=tr_acc,
                        va_acc=fv['acc'], va_f1=fv['f1'], va_rec=fv['rec'], lr=lr))

print('\nTraining complete.')

Training ep 1->8  |  5,919 batches/ep  |  bs=32
Estimated: ~148 min/epoch


Ep 01/8: 100%|██████████| 5919/5919 [18:38<00:00,  5.29it/s]



Ep 01/8  loss=0.6375  tr_acc=0.7212  macro_f1=0.7877  real_acc=0.6598  fake_acc=0.9429  lr=9.62e-05
  Saved epoch_01.pth
  >> best_model.pth  macro_f1=0.7877


Ep 02/8: 100%|██████████| 5919/5919 [18:12<00:00,  5.42it/s]



Ep 02/8  loss=0.5280  tr_acc=0.7869  macro_f1=0.8487  real_acc=0.7662  fake_acc=0.9456  lr=8.55e-05
  Saved epoch_02.pth
  >> best_model.pth  macro_f1=0.8487


...


Ep 06/8  loss=0.3700  tr_acc=0.8601  macro_f1=0.8715  real_acc=0.7998  fake_acc=0.9553  lr=1.55e-05
  Saved epoch_06.pth


Ep 07/8: 100%|██████████| 5919/5919 [18:11<00:00,  5.42it/s]



Ep 07/8  loss=0.3467  tr_acc=0.8702  macro_f1=0.8923  real_acc=0.8399  fake_acc=0.9532  lr=4.77e-06
  Saved epoch_07.pth
  >> best_model.pth  macro_f1=0.8923


Ep 08/8: 100%|██████████| 5919/5919 [18:10<00:00,  5.43it/s]



Ep 08/8  loss=0.3376  tr_acc=0.8743  macro_f1=0.8868  real_acc=0.8283  fake_acc=0.9548  lr=1.00e-06
  Saved epoch_08.pth

Training complete.


In [17]:
# Summary table
print(f'{"Ep":>4}  {"Loss":>7}  {"TrAcc":>7}  {"VaAcc":>7}  {"VaF1":>7}  {"VaRec":>7}  {"LR":>9}')
print('-'*60)
for h in history:
    print(f'{h["ep"]:>4}  {h["loss"]:>7.4f}  {h["tr_acc"]:>7.4f}  '
          f'{h["va_acc"]:>7.4f}  {h["va_f1"]:>7.4f}  {h["va_rec"]:>7.4f}  {h["lr"]:>9.2e}')

  Ep     Loss    TrAcc    VaAcc     VaF1    VaRec         LR
------------------------------------------------------------
   1   0.6375   0.7212   0.7913   0.8151   0.9407   9.62e-05
   2   0.5280   0.7869   0.8495   0.8599   0.9443   8.55e-05
   3   0.4730   0.8125   0.8495   0.8605   0.9494   6.94e-05
   4   0.4329   0.8323   0.8482   0.8618   0.9678   5.05e-05
   5   0.3985   0.8469   0.8812   0.8859   0.9427   3.16e-05
   6   0.3700   0.8601   0.8720   0.8792   0.9530   1.55e-05
   7   0.3467   0.8702   0.8925   0.8965   0.9519   4.77e-06
   8   0.3376   0.8743   0.8870   0.8919   0.9535   1.00e-06


In [20]:
# ══════════════════════════════════════════════
#  FULL EVALUATION
# ══════════════════════════════════════════════
best_ckpt = torch.load(f'{CKPT_DIR}/best_model.pth', map_location=DEVICE)
model.load_state_dict(best_ckpt['model_state_dict'])
print('Best model epoch:', best_ckpt['epoch'])

val_ov, val_per, val_worst = evaluate(model, val_loader, desc='Val')
print_eval('VAL', val_ov, val_per)
print(f'Worst-source F1: {val_worst:.4f}')

test_ov, test_per, test_worst = evaluate(model, test_loader, desc='Test')
print_eval('TEST', test_ov, test_per)
print(f'Worst-source F1: {test_worst:.4f}')

Best model epoch: 7



VAL
  acc=0.8931  macro_f1=0.8930  mean_prob=0.5165
  REAL  f1=0.8892  prec=0.9496  rec=0.8360
  FAKE  f1=0.8968  prec=0.8467  rec=0.9533

  Per source (sorted by F1, real sources shown with real-as-positive):
  -- Real sources --
  openfake_recraft-v3                       acc=0.700  f1=0.824  p=1.000  r=0.700  n=100
  stylegan_real                             acc=0.782  f1=0.877  p=1.000  r=0.782  n=5000
  openfake_real                             acc=0.816  f1=0.899  p=1.000  r=0.816  n=4950
  openfake_flux-1.1-pro                     acc=0.820  f1=0.901  p=1.000  r=0.820  n=150
  openfake_chroma                           acc=0.820  f1=0.901  p=1.000  r=0.820  n=150
  openfake_recraft-v2                       acc=0.821  f1=0.902  p=1.000  r=0.821  n=28
  openfake_flux-realism                     acc=0.871  f1=0.931  p=1.000  r=0.871  n=139
  openfake_flux-amateursnapshotphotos       acc=0.880  f1=0.936  p=1.000  r=0.880  n=150
  openfake_flux.1-dev                       acc=0.887  


TEST
  acc=0.8896  macro_f1=0.8895  mean_prob=0.5179
  REAL  f1=0.8853  prec=0.9473  rec=0.8309
  FAKE  f1=0.8936  prec=0.8424  rec=0.9514

  Per source (sorted by F1, real sources shown with real-as-positive):
  -- Real sources --
  openfake_recraft-v3                       acc=0.700  f1=0.824  p=1.000  r=0.700  n=150
  openfake_chroma                           acc=0.778  f1=0.875  p=1.000  r=0.778  n=225
  stylegan_real                             acc=0.778  f1=0.875  p=1.000  r=0.778  n=7500
  openfake_real                             acc=0.808  f1=0.894  p=1.000  r=0.808  n=7425
  openfake_flux-1.1-pro                     acc=0.840  f1=0.913  p=1.000  r=0.840  n=225
  openfake_flux-amateursnapshotphotos       acc=0.867  f1=0.929  p=1.000  r=0.867  n=225
  openfake_sdxl-realvis-v5                  acc=0.884  f1=0.939  p=1.000  r=0.884  n=225
  openfake_flux.1-dev                       acc=0.893  f1=0.944  p=1.000  r=0.893  n=225
  openfake_flux-realism                     acc=0.895

In [ ]:
from IPython.display import FileLink
print(os.listdir(CKPT_DIR))
FileLink('best_model.pth')